# AP Model 3D Foto en Kaggle (gratis: GPU T4)

Foto → modelo 3D con textura, esqueleto compatible con Mixamo, animaciones y
exportación a GLB / FBX / .blend / OBJ / STL. Es el mismo estudio del notebook
de Colab, adaptado a Kaggle.

## Antes de ejecutar (panel derecho → **Session options**)
1. **Accelerator: `GPU T4 x2`** (se usa una T4; la segunda queda libre).
2. **Internet: On** (hay que verificar el teléfono en tu cuenta de Kaggle).
3. **Persistence: `Files only`** (recomendado). Así `/kaggle/working` se
   conserva entre sesiones: Blender, las piezas compiladas y tus modelos no se
   vuelven a bajar ni a compilar. Sin esto todo empieza de cero cada vez.
4. (Opcional) **Add-ons → Secrets**: `HF_TOKEN` con tu token de Hugging Face.
   No es obligatorio: todos los modelos que usa este notebook son públicos.

## Orden
Ejecuta las celdas **1 → 2 → 3**, luego la **3b** (prueba corta) y la **4**
(página web). La primera vez tarda unos 15-20 min; con persistencia, después
unos 5-8 min (los pesos de IA se bajan en cada sesión, pero con hf_transfer es rápido).

**Límites de Kaggle**: 30 h de GPU por semana, sesiones de hasta 12 h. La
página se apaga cuando se detiene la sesión: **deja abierta la pestaña del
editor de Kaggle** mientras la usas.

## 1. Proyecto

In [ ]:
# 1. Proyecto, rutas y equipo
RAMA = "claude/practical-gates-go63pg"     # rama de GitHub con el codigo ("" = usar la copia incrustada)
import base64, io, json, os, subprocess, sys, tarfile, urllib.request, zlib

W = "/kaggle/working"
CACHE = f"{W}/ap3d_cache"                 # Blender y piezas compiladas (se conservan con persistencia)
APP = "/root/ap3d_app"
os.environ.update({
    "AP3D_ROOT": "/root/ap3d",
    "AP3D_OUT": f"{W}/ap3d_salidas",          # modelos y aprendizaje: aparecen en Output
    "AP3D_BLENDER": f"{CACHE}/blender/blender",
    "AP3D_HY3D": "/root/Hunyuan3D-2",
    "AP3D_TRIPOSG": "/root/TripoSG",
    "AP3D_CLOUDFLARED": f"{CACHE}/cloudflared",
    "HF_HUB_ENABLE_HF_TRANSFER": "1",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "GRADIO_ANALYTICS_ENABLED": "False",
})
for d in (CACHE, APP + "/ap3d", os.environ["AP3D_OUT"]):
    os.makedirs(d, exist_ok=True)

origen = "copia incrustada"
if RAMA:
    try:
        url = f"https://codeload.github.com/abransitonena-blip/digital-clock-timezones/tar.gz/refs/heads/{RAMA}"
        tar = tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen(url, timeout=60).read()))
        n = 0
        for m in tar.getmembers():
            p = m.name.split("/", 1)[-1]
            if m.isfile() and p.startswith("colab/ap3d/") and p.count("/") == 2:
                open(f"{APP}/ap3d/{os.path.basename(p)}", "wb").write(tar.extractfile(m).read())
                n += 1
        if n >= 6:
            origen = f"GitHub ({RAMA}, {n} archivos)"
    except Exception as e:
        print("GitHub no disponible, uso la copia incrustada:", str(e)[:100])
if origen == "copia incrustada":
    for _n, _d in json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9V19v2zYQf9enIJSHyq2myLKTeik0IP+WBGiWIHGBDUUhMBJts5FEjqTcZEM/1973yXZHSpHcJmn7MgGJKPL4u/sd73hn3/ePmGG5oaSk5OTyHbnHwdX+OQxYyZcM/hPJ1IKXpKKa0NII8mfDSE5vWOR5h6ImTU3J/jiOScHILCYnB7B3vPsaBzCDWJqRRtMB1Pn+72fnF3ueEYXQpIS/ShQM3yAK8AWtQYNaUlxmdWectXLN/grtFsm00B5skIpZ4boVRZVBTvMVQwMkXfIaTH9Ftgu23tarauRYVrQsKUBRz7A70yhqOZM1L5ggmpYAZhUq0FM2OQeqrQto5Pm+7/FKCmUIGNGO9KoxvPS6T6nd59XFxZykIBexes2VqKMlM4G/fzk5ynDND4m/nYvasNpsUzkp/JF38W6eHZ1dPbUNlr/alYHNvKAadh+8Pf7t6PjJ3e3yBsIN8C2Y6t4AskWumBSaG6E4uDsvRW3PIwDHHIqS3hCmjXN6h/IGP85FQcvQTgtpRt7pH6DyGS64vmHKaVPfN7SeHP2UgBnzq7PLi+uT5xBakQ2QueJSwFwPcHl8fXH9DQgnBEAgJKlZRR8Fr4OBCaACws0oymtWZJ8YX64MeHzkeV7BFiRbyiYY7XkEHqPu3QCfNiTAl/mqn1y4iShvChpxndE15eDYknUY3SPB7oEo2J1BMPOcZVIJSCrDmQ7i0cYeMLNRNZFRTSsWwtsIQ8usYpVQ92SbJC9fTmK7g93lTBpybF8Q571uSbX2BmD+4eU78E4cxS1hRaus4Po2qHidLRRj2fImnbXWQ5IcUiUZXC+QiOC0G/pRYGDYDO3TUXOyopB8WlJIszf4XYuQAG4uIsy0r9yJmsAlLuEiNCCDG2bJAr9D9UeRFRrybH1u539JycDkTW8XAP0A5HKrNd7fkIMgqegtK7jSQQERf8e1ycRtOlcNe/QsQAgVfq/TCxetfSDibQER2BnjdHzbil57f3CFvfe7OHMhsobTBJUuiN1OO+EusghyxjQPERSMXEBteDiXjXZGwyjLRVODCiIUSezyJ6FuQzi1KmsP8CF8IIHa07FGwOm8jvecO7ZcdXGlZZucwoc3zIuCAw9XWFLfVRZM4NyAinRnnIBCw6ROd+KQ5KumvtXpNMYn3DgiF1c0Zzod79hlAnUBRH/eDcGDeCmmSTydhVABykzU6CXr40dgxGJRClqkv9JSo1+xpGQKkeMZ4GpayRK+kmloXcKUTifO4VB0exdMZnsDB0zRAU8z3387H/CeoqFf8U6+wTv5/3knj/BO4pb32ylJps+yPj8+OhvQnsymHe1pT3v3WdaznrTj+RRpx+oHWL9OBqTHu4+Q1mzvC7AtMp+S8c6zpA/2r88OB6yTnd2O9aRnPXuO9PQJ0uMYD+cHSNtI+B7O4zbFtwg2jVWTr2jXHC4bqmzLJySHDgtqxbAlxHYDmj9JoUOrRYuxFuWaKWzQGFMoBbKuYEAXhz0hUxWHRsC2bO19aZEPXH/TwoAA4pagiATtku1VUeMrtC8EnWBYJQEN3l0vOoosgnzv4xVmm83MGqz9D+5ew0DejTspaBrQXQ8dw0BoGndXXzsBaTgoBu/91oN2T0XvguFUSNqLQ0aNLKhhAR5af51jOVZwDRcBfoVkPArJxnQ3u3nCeI+n+M+dH9zRKt28vHuEbgZhNuqNfCg1Olf8hgWyrTbterDwsav/W75/AUa/+PCZBDhuzcbvk4MR+fcfGya4MlzA+c1qvPBRBo1GCTxAEGl/cOCKG+IazLvMsfNu2M5/Cbmi0ONaMZs3Lz7shZ/xpwlEFUh3PxxwHcaIIe8egxn0Pijb+RQ2QBH/D5CQ/L8=", "engine.py": "...", "blender_rig.py": "...", "escena.py": "eNqdWW2P2zYS/q5fQShfpMRmvNlskyyq4jbpbq9AmwuS9oo7wxBoibaVSKIiSmtrg/z3e4akZNnrbYILNpZMzgznfYZj3/df57JMZc0CnZXsVpaNKEV4yURdCNaWgkmdyFI8TVpRN4olCkCizpRmhUpljmcnvLUsZS1SpbnnMfxbOprTJZu+cxR41bHpFH9ZIwvNzCf/qEFvOlVtwxJRV7IR+CbKZKPYBd5WqkwVe+55e+hLNv/ir/Olf8l8zvlTy0UsyqwAA5x2JswvVbGsJcFotRK0IvJG4fuMv/o6YUBceN5j9kakwsnBtCRGRS6YYLplgG9rwWopctbRpqhUB22UTObYBwYHgatEAV1dMlJE0UoIrtm6FpBeQ08l6asBmVxoVola0nKQSlaITtU4qJClqsMJlEYEkk2W4IETobCmVhNGNpFlAtlqsBAk4iMIfW4lKw3DjUpwxhN2NmNJwUC3ElqFxNgbY65LVmUaZM77wycsb+8IUCsSCy/O4pxd7ypVN729SZHgKiB7E/NGv0mmSggArGSvuHDSoxirg6r7Stbinu/7XlYQabjUuiI5+u8E0L8Xotn070r3b7rTXv++rDpvVavCgLZNlsOH7M6/ZdKo2vNA/5ZFhMTpdd6/8Azm2AX+dOqHpKzLBctWjL4yqLeHgl2h0fnCExWI9Lzyq3rdwkrNO/pWByG2uUjTWLh1Imu8E05Wy89tBj1Hf9StPAkJR/8uOBMCgGy6SkarXIlmArWvRJs30QWfncQxwfIAzvMHcExUnEZ5xl+EpFJN2qi40Qbh6oC0FXoI9kJ8kmlWmxXNIRtcYZfpJlafnGg21iNjag76aaAqWVp4sxeGngfLclVpvi04wi2NV4Ls2cVaNk1W4sAWJ8uiajpHlfxLgiohUoDJXcPNmvfXhN1M2JU1IOxKapzYd6Me905ie79cvQPcjJ/NPM+D1ORYQSkKCQOtl/hQ7XoTzfjL8NIktcKdmIpGQPRG1pnINS/l1mCFFogTs6WiQI8YsWsTIn0rwWdQshUivyTPKzjBxU0tJbcYcMuSkzFYFDH/9Yefb+J37399++bXd79d/+yHjhT8uWobPfdfC3jsG5Wr2l9wZ7f4VuQtKSd4bKQ4u4f1nuRCHOsTSEZmg1BL5D/w6HRT5aKUTjs6u8NnrhLSEWxeOAX1diyk3vCqzoqsyW5lbFBjeF5AiNGZQRUNcknU07Df8GKZVUe2VcuPCHG7xYkHUiYeboWytpGY6M9nC8vh/GwxSK+OjSYquGEaFOFYVgVZXZmL2NxK7L9DAkWEBORZSHXBbMLo7/CVPMevHOSMP3+GZX5+Tp/PLkJ6/BCaLN//64lTVo77wKUzrhzhG/aUPSNPxoOWKOnxKrOr9tQROfvPckEkLRsvDRsv6eMFzn+YgezuM2HcuOOnfw3HPHj+6PsJTu6HydwxtniYCzQMYy6+g4np/88Fyv8jNBhoSGwREbUtXKj/9bi6eSjIgoIXhYECN2sock3qck4vsUzeh2wVDCdaj9XWvZq6u/TGTFGQ2GNjk7ZQbJtVsMpyWUGgKGvmpsNZWHS5S2TVoD7TA3HCwI/cE0SgUS63DUHVol/qJYI2s4avpdk1/RDV6nDApPDKSpehkMZISmXSkyIhj4Wh7KTQeBgNWLEXBvURtUTu0BTYCbI4kSmE6RU2rdSKBb9Ck9VG1jKkZoT0q9EZ1YQrNVqaXFK/smrRSBqqjkDEvlRLnrS6UUWsNwK5ceCQeCam9inz6v3vV3/8+f7aN0DVkqAUrxQK+ZJ6lyMfAfIR8a/92faEHNUswDHhXt9GC1nZ8wfIYMTA79cf/umjW0pdpuIahmj0Nms2gT+oAH0IgZAuj1PT6CRnFhTFQt3KQIXeSe925hnAUMJV3JZ5VvZV2OQ4pRp9ZOFBfxQVJeyq2VtoiWQa1py9AWqNva7bSo0rYX881UH/X8uPsY8e68jv0H/vyPmIumXH+n2i8hzIcOqBDLEdmEPCwRQ1cWAE2Cun7hmMLEtDBeqLxm0mt3EuOlnztgKnMrAE0/VRdZFU/LBPGajSaN6rTUzMW/CqcdF/4BcQ9tAjjjzg0IimYKnRQUQ+XYe8UTEVy+DQsrqR1IIWYhdQsUS/BD+6lXWTJVKH7OlT9nw2mx3iEJtPyLrkSXW2i7eqzlP2DzZCnWcLCG2EyIxGRbmWVGyOj5gYFsLFwQl/w3+c5FLUTgrqYeAz1dhYB5mmKCGc7dmDoMjKoOI7G67EFPCooJrl7vTy3eFy6Dqv3ZgsdHeKrFnuTi+fJkutIgib5jjo3dq0zSHFyRl3hvhEPSfBPjXkih3oTSErv0MXIqc/hPvo2TcsnyaM/pxj7hC5HS0DeWeRdyF7TCC01NmlziwZDHtLHIkdEArCjwiEVBV7bKDZ5W60TMzRm5PUVLq+M0qzpAnWkeF3wraRYS5KuwlTq1XUX09teEf3o52imVR0vyg7d6UyPOD1PUu6AztQwU/oWC7IAH2NFv01u79a379WT+7dnm3p5hplKfgkuygXxRJ1Pblk02Tub/0FDsMLdQOeStoKtctGuu148wyyBDuER4f/O0Rid+aSM1x8N2M/MtsoQbEzPntG3rA7A+92cTosdgb05gi0I9CbEegotdl29AbVQI77U5HnAU54wujm8iPczRw5Az4t/BSxxBEeIJbu/AEiPbYHubyYMFwU0ImnplxaVYT7xj+RQeIk3xJQxKz+Jk591nuooA2JEhpKhmZrL9kjay10BTmzsx0ahqD9hCxo/0IaT8i1GkYlOVVFYWYqmZkvsVezgRjF+9gCF8POdoNWilEgbEkLBuYwIxO3vasHFtA0lSODnOORulYTQR6OsB+xFZUetDYiyQR1MPbacHDEjvIxbpfDIoX2iP4xuxTeW7KRdZW/4Xcs9DkejktLwLyO++LwYUJ/nZR2RGf6MCFA7cV71KeiQOoKlsphTQ3LQjOFTDNkihZJVaH4DBqzGjyq8fMMMQmiF8dF6vksHLmRcdqDCnb+A33Mwr/RWg3SRhykBXuTgEdlotSBCM0VjVQ5AGlUmlNAs3DPcoKcmHTmAkvM0Gn787cUKhQRQYr8GVJEiKU7mIjTZTe0mc5OngJC2Ot4qci/g4QKwdaZIyHTpM5MiXHb/c4TtxOO+xKbxB6D2JFqXJT32iGAg/21jfK1f9gErHl/W49lm8uaWLTX4OHyPpywWlkaePGp8ai64B6EJSeDY7pHzPB+YjCohHB3TiP0DjNM6Xl3iOky5zCHOUituO0rIoy8MrEXD5PAJv1dLzFWpSJidRcM4Obi4BKjtZ5BD3t1QmpXC1FbTpGy+bGHMdcJIjDqmIzu+SZLZVzbYbobJ/U1MW/vhsmZbZ7RO683jZm0mIGe/+HPt/5o2hJQRJ9PGEqrR/Pfk0MWbNi+nkb6a8pZ53xmVu9b/iBALmgQ0k8G+sVzipdvcHn1/vrqgE262e/LpZmCUFZ4AVK3sjzNNjaO2Z7N9qs0EMLaGb8wa98Q5ZUTZXaP9wRXuloY5o/5fQzqzy56Zl+BWQCfZhYbJ1jom7iZG3oQmXPkiCkj+P4403Sj90k+xZ9bGvhM/0sN1398s2GIBW5C6rilKBTF4DfSzvlrLytX1PB88W3xigv/ks3dABVB4A/ejvVRoMBrYySPTNaq9N2Ffd/muc6PcOZf9j/DjJydfouhsaw5r1atDRfTTtCQxzd9weFm6jZPN5N+ZY+z4LcEadLzrYk5G0e99hZf2WE4Lr4OJraKOTmRUZrTk39UWTkadvv730rMaMUSMIOBJvJ/+e21/1D/a+c6Btz8skK8aXNbH8iYMNHuBj+MkHpmiTXwBC+grjD0jodEFroSWo/H61rcyljouBAwPih8t4Tmpx2SEY14VUvd82Xm+mlbVAF5Ey4HNN3/Bi3CIVKwOT7pt5myiRDlstRtTewlWRaZDB16brh1/eHN9VtKEsN52h14Ain0/gcR1IwJ", "app.py": "...", "memoria.py": "eNqtWc2O48YRvuspGjQWIjVazcwufFEgG47/EmQdA/HmECiC0KJaGkokm8NuyqMRlDdIgFxzCXIMcswhQHIIsPtCeYR8Vd38mxmvvYYXgx2y2V1d/VXVV1U9QRB8pTJdJlKsVSqKUh9VbPVUbCtZrqVItbitlMCnymAk0UIWZaVWUmgR67JMtkocaVZl5KCQpRS5FqUqlE1KocpSl8pMBoOR+FRCGl6q1Mq1FkWl1grvcbLCxP/99c9/vMR/fxIqnwqbFBraWCiCqWOx0WUmx8KqO1thg+NACGWgVaqsvpR5ksk40bkIY/wX60zllhXVBQ3LNJpg+1fSsL4qpkFlIN/tExaqNJjmBKWXsVype3mpVztIj4RRHgkSnYtUGuyeqXWyliwjlQKrUjkRP09VvlYlTRGpAipjYRKRpmorRZWLTANfzUL8cggCYEAAz2PCj1BmnR4grtJki1/hAXOBbkZHUWUsc90cTRlAlRibxHgJaX2mrS4vi1LJXWWsEkamGFQ7XfKhsgpLSpxK5bRVIXOVTgbf1Kel4S+13mLNZ2VyUHQU2gNiczLKz2ggh20UYSJiWcLi5EK0Dx1oMgiCYJBkhS6t2AHg+lmb+sneQLl1km+bgSRTg8Gm1JmYCD8GvDbJdjBYvvr601+JWbto8krH+zAafPH5J6+/wYcwkHl8o4OxCFJZbvlhJY2i30WiylwafpTWPaxKea/5KYYnpTqIBoPBWm0Ahb0JoymsI8Sazz4TwSX0sHCsSx65/OrIqAQ8KdngUBNaNknMOilDnuMl0L9SwW/zZtJOJ7mbg81l8XK9REjBd5J7uVMTwgq60DosyOQezlKa0OEw+fq3r5ef/fI3wP0O1l7q/ew1otFNf2qbh8uCJ7Zyx061XNfHtuWx1f7bxN4gllQeOmQiARfbtN87W5PACQvaOJXUXawKKz7nX4i7R5icggP81ARTMV9AO/J+93L2ahl5UGHm9SpgCqeEUzPjAXEhggmenTVadTEEkd8GDxVmLddVVoQZuGUsEoRtbmfX5MymKtVSmjhJZl/I1KjGEKC0VMbKCS1q0DZK4hgK5kHUeyX9yeYbAGH5w2SrbLgfiytgBy4Te2wp2G8XXg5AUOGaVNkgpmCibYUwGwuNVS2jzYLA74HY+tLFKZiFEOwwKGIQgw3TTigOaY0EViSftQkcK5D/n85RCxyHWQtVhjXOMdqhuTfZYiILwLwOTz1XCGww5UiegF829BAGz373/Fn2/NlaPPvF9NlXQQSzODrE1NpdKVZzidnraNwXWMOByS0ygd7jfaV1Guo9CWxhwnj7Mp++vLpaPJDI3Ihp0mHhXiHjpsqPlcxZQWxllF0W0rBH+qnArDDBQw1Bf7IzaQNHMSyE/BnjYYs7+UOyAU1TGoJrwh1gAPeNZ/dF/zT/2u2Rr4PogfofCO8MBFtBSdVMBadyRxbIaZRQ8HGV5C7V9vJJH4vasabitK8R2Xf8vsaQTBjbUjFBO8QIMHVHv7YVckgeq/dGgwQsS1lgOclBEYHkzFs0Ri6RThW4Ozo/sKIh31qztVvA6sGl1RDVtfy5jQlPUj6YrZLxzdIeCxW6eCa7Ll3xgZKiieBPfflUZ30XuC4wpgQ5whugI//vFOyinU1SbZB7ARAcHKVVHd3kV70Af+BodabKteW5j5iY6e49icDRdUsEwQbY9TmxDYIeCu+0KzK586CGPJ0MFvb97PKEYTrnpHxZ5xYrrVlmTdp7cMgDXjO3vWO8MfJSD8iDkPman7JWTRM8Tv1h8Mnbf+dUGt/IIxO2mYjPlCmqt3/nMjKm6tiXiMCu0Ag7MRq9+e83KKfe/lOsYO6PRyMRPMItOIpO3S6S8u3ffNxiCdG/0zhNqOidIS0Fo9EJpWp4iM5OE/HmX4JH3CGGfIghHxZTniya/Qao80ajYMEbUIDfugB3PIaoprrdRzVV7vTYFO5dlG5ZszuWcUcyDoTw3byl/oWYzcTtolmBz7emX4PoPYSYKguxDslh0Uq7NVFvJmNRO+0meC5Ot5MYnIEAT+5VGJ3BXXp/vmRQsPjM8IvwdH11JUa00eWlaL49izzEH4im7u6wI1XIDESDkpvBSNUJhzAqCfbtj4DF48y+eNfNaBF95ccecF5z8dFMvPjJIPyCdODu5sQ7EoZPw1WD2YywE9YgLscetplYoeFYIgdbVPsm7LhxvW/w+zxA8eeaGv68srTuyOTLgV1H68q2J+3JWFk3B02cfhJuiu4ajg7CnXIjWsyffzhd1FuxpO/YDRqPRm//koK4sBuCdpUqaufIcdDOoTWxaoqQivrLxQUHLhz1bj6sjT9cEMYYaFXBUNCqz4osutxHgLm+gMXWCQvxrbOM9EMJytbjZtvn0bHwWbTNXJ/4eoETUqkSbo0kvJ404wzFNp1i5QHRjhYqjCWqB22ocSlSn5WiJnv9cObNoPiTUVE7PpeV7PmPK0uqtlBDLxqLQ9LpFlVKfgy7dm4hpN2epKJ+OHTI73tI7+yydY3GtzdH0uFMB+w60NzLWZDKnGCgSetUfv08yCt1kAiSLElTVs3lt6Z3uzk2rqfR7gs/05cS0jFTx90+8OzETMZxyEyGgLWVt6oIoQv8V7z073X6qnLdCiqlVXyyQRcjrPoBvPcTct+iJ7RPfi+nj5IpaT3HqnfQoGi4rFktY1vJlK4J6sPQTihQQtYlQq0iS2uorgqD1yWXMkjaRrUdRyMLViKvbD6QpFp+dwO3vH7t0jtLgK5sAcKFD0WjC/ERvzBOTig1paDQq8n1h300Gg9zcBIiJKI3p+NdGzcPeROTziLsNQigKVBE2NFjJJAbmmQQteWp8/06dBzjtlT0RAB06n3Ssa1iH7p/fXvoexwpQgOQ3vwD9RKtfvOfR1q0Uev08DQ47QQL5bu1QntwqXeIhA3dBPL9nMrdtSDV6rQfdVRge3AkijUQUhLrx2epu5V3H6RRy4sW4YuX4qZSpk2inu4blqE4c7wPQdxz9tgg8HlguVfHUO769xjNbYkJuaHbvX9H1zRk79vMPdW+Na0h+jdhdGlJa+Ovwdw5+oUD2GrJRDV72WawV7LXz4LC2h44ZwvShYoImV4uvO9cNNBHdC2TszBTyUNyT1faK3lUJkEaF/U9OLgqS2DzayiKZENaUNlf50S/pbvZ1m02PDygPpcZH+XDTtqrm26f2rZlVXA14+m3JdHWebf47OZNABVgk1VqQ3YB0F4tcIGTntxdyxXAz/1vucNDd5rvzOaLTvO1RXqCL6N2ue6OMaNeUKtqa4KNuvx112+8HyYGSLBtywkBzUQvJqZYBYBbPvaWju3PeZApwiRk4LxySAONfyx6PTKkPGrjfo0maCw2rplDsFuZ15VOruDvCYLy1Mg7u78YwBBZYqje82hNAt8ugIMTVVIKvRDXlFhCl1AvxItoStft9V82KBvDncD4G/47jSh0XBdZru7FZKqZ5F1Iio8FzDhLEf5Iy9upCMMGd79RbRvsNBbPKdc5XIkb5ldo/KLmwtVV8bSDl4Fq3r3lvvWTO1+tk0P4MVJnE1Br+YcTiefPQztc1Am0HZhOrjZnYTjVOSmkiEtwQZeK5G6MiAxeoxLhvKIaSJFhiqa1wM/JSc9REDuUTvb8RO+8CUK+3xMnT2tDZrIhUQvf6bUfmM/4g0909Qd+5Q9Pya+zzmno88yQE7pf23Lj0BcEwzo9Db9DYMv+p6GjR1BPT6gnTTTwzIlerssVw3PUXPh326S2sucs5Tkxbf/MBo8rm1sqf08RrvmWokl6P47AehUdZ5M279aXR57W+GLi3axGbsfTuqR2171D4uz3MSkAIrvqUI+F338nL9n5dYfJPEfwRo9Iou+z3PF9/gjGKd3l0B8uKEJ8bm4r5eC0P9fts6R4y8m5w1MO87m8i0BAY5ZHdHxKgmodumMnVmWguB4D3E/F83scAD+I6/8DIpT/aw==", "estudio.html": "eNrVfdtu3Mi22Lu+gpa9TVJis7t1s9RtyscXeTwb8oy35dl7H2grcjVZ3U2LN5PsVkvtBs5DkAsC5CUnFwQHSeYhBwhONhAgQc5DgDxs/8l8QT4ha60qksVutuw9c/KQATxNVq2qWrVq1brVKurxPS9285uEa+M8DI43HuOPFrBo5GzybPP48Zgz7/hxyHOmuWOWZjx3Nif5sHW4CcBUHLGQO5tTn18ncZpvam4c5TwCsGvfy8eOx6e+y1v0YvmRn/ssaGUuC7jTxT5yPw/48UmWTzw/1nZfPG6Lko3HWX6Dv700jvN5qzUY9e53d7q73cN+q5WwiAfwPuh6O3vwHvgR793fYTve7gG85rO8d58f8iPO4S2cwNsRO/IYgzfmur37+4P9wbADb/EV9DI88g4Gi42t+SCetTL/1o9GvUGcejxtQckCaWINYu9mHrJ05Ee9Tn/M/dE473U7nV/1B8y9GqXxJPJ6U5YaiKnZd+MgTuU7YGP2h0CWXncvmWnZTZbzsDXxrRZLkoC3RIF1xkcx13741spYlLUynvrDfjzl6TCIr3tj3/N4tNi4n8fJPIkzoGMc9dggi4NJzvtQCsgks37Ah7l4SiWG8Oj5WRKwm94w4LP+iCW9QyhkgT+KWj6MnPVcWDCe9m9bfuTxWW+/j5Ct6xRA8X+LDTvjo3mtm5VZ05KYfUG3XhcnGge+p4laXKCispUyz59kAreEeR7SezcRqO0kMzGcNpjkeRxZ9iCP5spoeQrkSVgKKNeoDKtcjt5ZGumRMhA8a10YRayIH42B0NDTJM2gqyT2iRLXYyBMC4ZxeS+KKxpIpOy4htL9HXdnuMtWFr3WpgfkY4OAe/MYuvXzm569u1+M6/EhmwQ5NkjmSN9eF549r1rqlAcs96ecio+XifL1S9CAo+dpdsijSbnAURzx/iqTCY7qCGbrNnH+L+SBA3gO/UgIi173COtCNmvJ3XbQmY6rHcEmeUzI23HCo6UZDILYvYJaLNSYJX7FOtRh+nIs3Mg5n+Ut2hY93Eare5gAPO7GKSPKCDpJ5A+RrxDj+gRxTgqdqEnJpXewYIl8b4xTrk1BFDVwYNEoC1kQzFd2B9CN5T1aRpAkfJas4aP4qlwmtQ8sRoxb18V6dKCbQQwohQ0ySVT0aK+RWNpHGuPuHcZpKPYx8DT/vdGCGlORPjVBQ6LI81PuUveA0CSMmqRXIdiQY8SqHh1MrwFDFvlhNl8RgrgydTnX/zDJcn9405I6TPZcdFEw0C/ed4qoKthkZ2kffFFELSFlN+EFqq5xIalcFNwfDofQlR8N44YlFNoElBbt+QPEcZWpkCNAaXKx0sUq7vVx3sXW7dr7MEqWsxEfMG/E59jfPq6WBMDdrKzcow6tXMBHPPLm60QRDSjZ7NGSsNjbXxYW6jaUloPr/mxZVW73pfnXZGhJjHIuNsg3zfOnNVlbzXRljUV5rUWNgRv2gTRSABmtsyg68OeFoKto3iCtdpUZ+BGtoBSl90F2NK2EH4FNCKKshl0DVs1ba5WdCpLtLjZcFk1ZtizT73NvLafuSE4Fca6w4qEU8jvEIzUu2O8e7uxxvsoF93cGex477G9oSzywp/BA0/rLbeUeuZyjGQIG5Zh5wIIdraPhLtfu74K1ORzu7tJcNJt7+VwVq486naKXw+GgMxzKFW1JTj9AC4laolEmVxtFRmfFbEMoKR+S1K8pDIFEXQpwb8yDZF4Mzo46QIH6Nt4jsDArBm6RYKjmfeQNDhBjqCnaEL5/EXLPZ5pRbfGDPSCGOcflpMU7KG1WfBIwtG1xBLF/pULpHtbJ3q1vy8Udo21oZEAXuwe7PizU02GJAHayoa0Yoap4PqqjsEsthBhdxrd5ANXakHuhPqWiP22QLgkLrdC5coDdjoDO4zjI5nX1V4rA1kxaTFpdmRWT2m+QZ3JWS2K7thb7ndqkllalwmV/H3ApxtqTY2HnUvDV+mvoRTLTzgFI9arXPZphYU4f7Sw2FhuP29JxfJy5qZ/kGrq3zqYfonsasmTzeC5fss3efDMfp5xv9jbHeZ5kvXbb9SL7Q+bxwJ+mdsTzdpSEbQL6i47dPejYnfZg4geeKLPD2JsEHFpsWqKrNswxjrL2n9cln7EQXMGs/SEL25uLBUyCkIdZtIULjv4nvIEi0HzP2YSFAO9Z06jADViWOZvAr5tUic54RtUAINZZ81jOWlNoyGebRYM42jx+B2w4SdnjtoA7XoYHTwsIdfzG57csK6EauxbyC4F5Fmdre4TBbzaPn6auHwTrxx37ABPk6wE+TpgHYz2HnzSuYfa4DURZIQ7Qq7HC8wTRAkR+sxxG1sK+3zz+6d/9lXY6udV++jf/u0JH6QJt7iZyBxhEocDGZhHhaJ4OAs5ABPhxCpDySTNAFZl3kBxawQ9oJ5dDs6fF49ohvJSF4Li4bPP4BTx+/hGf10KDwIkA8g3+aEbKp+C+plou2KVCS9D0Tsq6LAybCft//uO//M/a888/gi5hfzZ1XWezY3U2j19iKIA3zwNguocEdQLOfOCxtWBHBHUKqlrzbz/ad5AdgFsKNBgH9tped/et7uHm8W57by1Ixzrs4HZI/cF67IbM5cDrTN2oXyY8eHjNdD+ZofyD5byT5kUfYgFq4xU/hThC45DwG7EIaAJSkQfxT3/1t8twlSrZ1EhMA7JfE3SouRb7NRtOhvIqQ2bVjWpwtepabtlOQwtDkz0fHpaCo5ouKa3Vcu41TmtFUoPJh8z/t/9DO/H8HLa6xzWefZzwgOdxs/gCWwTsrqZ9EMJyoZZXxbpGQVRnE9iKZeBpaxPYxCDJfXcSgLb8/N8jjWnZRAsmIway46e/+Rfaa+zjDraHYWB1ALro+xsf+h1PUNZrCW5gqtZ4IJcfZ/jX/1h7g6V39+syj4OsKTo+AVHMAWPROfTtpVy70fLYZVXvYkg35TCkRMIAHmCWxoAKlpYAjAkz+w//Wns1qemjFQ0hVg5NX1CHOMgqsYxkEuWxxm4ngQmosDT9/CPSNWAY79ZgRA1sb3fMMvuO5RssLZ8YGLgYaPXTP/mf2gsOPNgkwQUgsdxPf/PX2nMOy5pqgmzNpBUtYEWA+X76r/8UV4Fr35w++7KeLDCt94SR6puSw8CZAJGVBKA/UqC3XJJ1aM984PYzMHXTJeFVXwBwKjbXyRc0hFe3mzAaN2v9kHG7RtWXBhJZyg2LkcZ5zTh6CqZuCwpZ85K4Oeyd56B90nUcjlDXfgqL8CYOPv9xFEcNhhEJxiseoJIqZUADTDK+yWrovfz8x6xJhRNuSBjCDVWUFrKsGS4bxySL/tXfa89Z0mgKiuXxVZGF+3wF14qlip+aBS5MZaC6sL61LQ12zbtXb09OtGEah5pOJrHeL+rn36cDP38e4xSCbKECFUa2KyvbNVAwtpVevjl99/IU1BNPG7sIqCprV2D15i/ePn3+/Ve0V+DqHbwrIp5fN5MV8NXZCO29Bh8uK8WMCtB6J2/jODyJpn4aR6Db8+Z+qvqsvdRA9LYBOGe59sDJnGMvdidUAzyR3pwBX7ig1ozM7Euol9+enpw5czAfe+e6EBeX13xgj4KBbhUFuHdBB9cLhcXp3xYVFxa5JNBNQj6JLJTOBxajtJel6GtAERPOhiwc+4gDg4JLBh6GLCV/okKOXkXVopjEd98+P3HmBcSE79vDwUzv6S+f/V6oox8i0EeBdhKN/Ihr+5ohNRPqiCgOBynPYDOCvISZfpx8/qMJ0xQgl7jBYIq43HEEfb6bZCCHfReMIdmo3HDQaOjjxi9gX7AcBhmSNEA9iJvd0tza5jcLDMFVbgs8KxqT8XQJ7AEI+hgsCm6g2zenf0mYU61myGpUh+EkAO2IxWbViaCYoMhrpK7GoS26ZzgvzQAqKdDF4vd00EoaanzAFQo1A3wTN9a2tV+/OfnGbGYO0UiWaAY0s7RvYi/OLe0dueUfMqUlmJDKQuGMSlJqBhHEkitnaa998MRjaLyhyeb2IADFC42f4S8HQogScz2LCvQyP6qtmQQGjxCdsNiOBx8A8vtnvwaonJyxWBm2Wo1dz87yAEDP3p1qyiLsvoCVDhU0gtjrXu53JAqn378oyLrf+VUNaueyu69AhZObArK7/yvCQdlawAf0AuadtLSgr2qP4VonDLmMyqBO3WyANHCJhvKMIQr1TVfwCRZoMC2w+KW5hbRLwTRy/VUOU1vWOCwA4vjkq+ZxAsOOPv+RAR9Jl1Ubx2nEmceww6nvcWB7FvoRS+0w2YMef4tlgqkIixJsBHJv96BTA4P3P/03goDlvEzQPWZ2Eo0A4NsQfJwIKUKVmqjUF6XETImPeOpE/FooQPt3fPDN6VtZbsxZhGkSPst6eTrhFi45T6ccdsY1ODbPJsMhT6lqAQK26M7OeP7Gn/HgLZ5OGq9ZPsbTBkNkYFQ11o5p9tVGZ+AGGX4U8fR3MksDHl+RPFW7jyd5MgElC7v+DA/GHYH62dtvnlWFVcdgEXBgjQQQlpBPn5+cvfSD0HffVXUKJuR/QbENVi0elDs4w/5GqVswBGZDK2jwfOwHnlE29UA9BRyBSoWTgfjjCoXP8N0w+1RuVw6kAkKzMDozkWVS9pSEKQ8VsDev3568/gZ6SRnquAIJPFFDcFCc7/jMoVY2KlYxMrZfUqOGaXXszp5pCw7lxYDAlupwoMsTPPyc8ucshEGN3X2rWq22slrYXdfa6XRK3N08oK5q9pEBA1iNtANwSfwXLKSVoyXAYoz6vgVLOOdNZWcJh/Xq2jvFwKPUV0n7Dby+AgcLmLvbsfY6Vme2w3e74mFnZ2/HLVYGzA8DG5dTGAZxrG6V1+AkGQp5Ahbxb3gc8jy9MXY6MH3TUpZdMhWG1FhgqLkXC5NOxOPURgsfow72zGnRtnnzbXunqOMuB9KLfsTUK0QJpMR0zENfQfQVvGbJGIh8iosDjDWk/2DKu/Df3p5lH6izxuYV+04ipasXxfE3C5b62oEeJpHtgpdfQxHKxH6yQ5bgBseNDuTZO7Twf+q4AFsOm/rh3cMeHjIGw3bVDqARdLABag6EMSzVJAisEMSNfGTURebMF2COxe6VuuPwHbbBIJ4ppc/i2S4UvgJ+OrTQMRKcI3pDTxC7YlPmC94+g4mVExDmOJVXFr1R7E0PTQuqVMx1qKVypNAL7oKqSd8AExj6z43fB/4ga1OX7VGQD9u62Rdo0QjKwASDpBtOIiKSRkFo4woPqjSB8qkzlzHk3nm16tb5jrVn7V5Y1YoAN1ldu2Ph+l9YRThZNtrxPGvXhmb7FjAANhsMvUNoZu9b9hE2Fc3QeS1jyqIt6xzApgKI89YeDEyNDzqPOjgm9Agbif6J9mWIWUF2F8Y437X2rVYX2w6H+/uPHlm4tjYg3d23hizIoC0FnJV29p513oFm+6KV2DrQ7oDGFY0W51cXfUCa9gCKbyTxKz4zTs87F2Jr4El6lMGed07PuxdUVEQZaU/Ytn16vnNh2mTUguV75rKApcYrUJJ+uNzr7oUoVnvdIxSwtNZxa2frlYX/Wrtb0Blu71qz/Qu5hxRXC4ofXTwRKqSH/L4K8q3SxQGN7A+Ne9jOnDeMsW0f4KlYyWED2GuGiYkG9iTxQC6CwEv92e/iFBQqLiJmPcxAalzxkzDJb8q9M1jZoTg29pOnbApKihuxczwHZGLbz1BKm8XL2RUqKU+UwT67IpM4kghARzGQOQTjgj9DlYwmDiLZH0BxcgNdDKpi06ZQlMB6D+rCCn9zQcEXDpwBw4ykUvhC3yXYVw+C9JlEQExjYC4WaEuB9gbKxjOFzkPYCdxgtw7sDR443R3a04L6kqKugz2NeP6czn8VpfZbcp+BxKaVFUBkpTWB9F85wtZjMyOzbyywAcTSCJGnVs4sBMjsW3Orax+1qSZnERoE9jCebhXKD6xbEzRg2Qlz2G1ZiQcq3OGBWkCgbGkDuPZsm2DAmjeYKeDBQTW4ueVZrn1T1cqS2+0SZhleTAmMDpBOQA2xepjGBKOCWZ86XnsHEydwJvi2tS9eBI+9SeMPQpWJNTWErVMyoLpD0D/LYIugPsudjjV1Ov272Pzhw3uxPQEDHdxwZmO0D7aXMIKcir36021nZLMcRh8AM2YVqVzgu7yfYzWdbzyRv6Kivdu7o1l7FxmwXKevYBaAfWDolFmgm7YwHt+9PnXeP5gTvckoNnIwSuPTGNO0z2DoaGSYCw0ePv8YjSYB+Fp/+nvtwXzaADT9/F8wcs6zx4P0+MEceA6AXoI14Bk7UP3532pYeNNUeFsrDN9v4242EFNbBEmevBfjKkULGfx/39N12P4su4lcrVxKPgMPLDOG5hxWQC5K6rBr5oNdyXN3bAytOSzPOPZ6+quTpy/0ajuDw3O1cBkCwYLKQtI6i0Vp7PzW59eODva73l8eOxvH1waexJOwJUB8qxbLZZGXORQYO8eaiz52OvQDTpZOH5xmQ5rAWjwU4CbwHVk+9pjhvMw5wQ/7g5Szq4VUBlhmCoyL9UYDBNabDqdseTbl6JgvpFcYjSRppLWCP09xUgZ2uK0/mTr6tgEeKJqVl5ewGT59AmObi+3AYfVyFpjKaOjXPJfXAd6vHg5qNZbrdjpbnMbkXrvobPGr94K/YQCgojkX6hBcq3jKqWQBNZWNuASgVPSXDMmFUF6w6ahF/25FFq9Y2HGTZxCDwzcBay18DiOAQ0Ts0ielBN3hIjuOo2NQUv/0qXgd+7pJGoZclCVP5wzEswf2WOXBiINOMOCOBq57YAHxRuOIZ1nP3t+38EZGIN46+4JytaFllFTHGX31iEATWEnyjmVAQhn00QJ0oBip8ghwYfrL6i6MSDqFfmTfELlLUXbTcsJI+ltVIYhddATrJdBp5Vr0K9eij2xHZz81qabrknlAhmJgh8DtgEejHFZV9lRO/2kB8horylkU24NOkZ1z/S0HlDAK9nSQ+qmWx171kkyAWCwr3132AQO4+D5l0S1L4QkPTvH3bAKOfYQhPnjBLAYqfQXuNz28lmBPM9g/Pj59x0eiYZxiSAgjai8DPJ1SOn3LB7LkuYhu4VOcgqePDVkw8Zh8ElDPQJiI4VhE4Ts8Kbx8p1+ImdfolsVpbhjMGpjOsUHUEJrqexBKNt4DMrePjsxfHR21lmoHtVpJ1Eq+uSjf1JHMuVzZc5daXji0VLYb+MlTqkHVT70UizNwyjgRHgrnXAY1DF2cTulo8qkCSfQMuxhEocsN/RLmrhFUHME44KcaME2Uk4YANWs8psahBtLwJOAC84JNLp4UTz11iuCfyF7F1KQpghEc6g11Acj4M5BaUoysk1CqdV20kLFBRBjlHuBb3D5x7kkYtZLOFE9BVxaCUyeC4Z7m+e/AojEQGE2bGjAeWjE/ygR4aVuI7IgVbbMsgyoJSAc6+hOdMnz1no45E3ofpleFuR4f7HXMWucVGihxdOB1fRWDShIsj/5EfzwoE9WO0Vh5zjwmzzqcMvvAxpoXMfAxkEIL+S0MG1lawDSKZWOhF8OGtfUescDylHAQET7HIxwXpEMSy6ybY32boE4JWcPsCWElHAezL0IBsOinIhArzdI+rMgkeTO+yXwXX1ExxuFrll4tK0dZ2pcPQmIidaBg7Zov1hsLYlUAw+bDvqdBAE0p9VArthw4FekJAwtq4BwPlBHzeDQKxIjWwMY0EJiVPQXCkcFUt8drRJJGXMa5iI+9ZgkqmJ9lnocOpsfcABz9GpVKNJ9Uz7BRe9WbVCfhw4chbV/oHXEhc0yUmCYVoAMkCiz9vr5dBBJGFEgozGVTak5pXZ7bto2NLzBoZxjnkeVegAx6/5gyr/witUjJfnowdxeY7OCDqR0txLH7e9P+EINPRQaxtFNRswkGUOiK0ioqiAEunhRc0QUy1T1WWJDC9MIeHj6UD/cch5nyGdwtj38/yQ171+wzG48qcsOk0m8jLLRpIFokQoN9iYfU/OSv5SFVsjtOtCLWgdSKhyeFWhxRiOIeKno5WUGM68w5v1jDVXVWugaPbJKNjdiElYSXAtlr5/jaFrcUi/11ba6NmNwrmPXTp+UBKjOeGG8lnGJ8DQ/3zqvni4qioXMc0igkcxwgh3R+FuVwJX9cL4fgK9fWqtc8YyCaVu3Vw8HhYDi0yuGEJanc5hQFSoDe8niSj3+X+jnvCcfLLJX+dZ1Mwvy+tsWpxvdkqO2AOY6q4Rp3WeWykTh1dBlU1ZXzpVVWLK5aLnNhYSGg68NBTsbJmxQQH5Fqr6LNzkCygLRFZHnseM1KFCwKocPuRknBZuYcz5okOfVEyuFebHrLulJUo+gpB4Lykyk8IBAebRk6zVC30Aj6B0LHvJPW5yLx+GIdrRGRebl+ldYIljXl4mvGcb8wTv3ESxhgqBrTOL/TXvLR2C1Qc9ESwNRd8Lf+7MifBM2d1UrXnlkiSDsD3+nVlt15hMGzwllZjsPlAJ7bN/DvdhuA9/dF8EsNpOVL8bBKDAgszpkFRrgyMxuMAh+Mawuogcrqu0k4QC9bhj0t/qVluNNSUNeC4iiKkYBsVLPxVOA1FmKhK6SRWC3kXYt+r/7euPyqEqqDmwsxjpunS+MUBt6GYlPX8RBSInLufYXV3W+EUfFCsa5oPImXtPzvGHjFN2gYeAVmeeCNUo2o8ZfqeFYeBcizYfK4K8BSZ9mkCd6BzJZbsQkGL6qJb07sqKebSghIbI87Az2/UJ3OQ1vRaEIrhbbUaQ4dOpMykocUXxus6v+/RrrAsNuvT0CqXcS54Jvx6sYpcwnEg0HTskAOFUKMrXXOmY5G4xhsAqchIwHYCVX8D29PDd3HtJo2pthgCy++jtBNcXRXZK1S8g1UEFoGcvmqSsPMqVsudNocxSSjzAqnOZvi7uODr02gQTHYbmst+E97SVl7mLQ09adxj1Ke0pzLjL2WfG2xEH780YR5cWoiNAbcZSahx7G3nIPHqZHDKnLkgagupgoyjaFvCion4CnDEsxDwix1jOiKZHWQ7SkDz5YMoixBTwTNXWvsJ9mzOBLRZ8w5mr6pHn8rH8Hv/D4ivrZqfqU8fr+cNqgr07pkzcUfleLfTJA56VgNatTUnZNJQKf0NQNe8YHn1ST6tUn0q0n0q0n0l0z+O04xsSdyEdrvgOJt8DKyHCqEmyfFZTANnG1DluIOg03VNv7gmQ/a5qdP5x2re2Gewz8hfSSqwmmYD3qxBe2tWa9j3cK/KT5M4Qn2G8rN9iuYzoP6uOBxFpM0y9nG6EjWzrB4SR8vF16OWLtPn+4VOIgwKBQU3Xz65OWPnU5BGC93qiSy3Oq2dzs0iwIcDRg6En0jrQ3jcipsTqK7ORfkv5zCpowp+UouwiorrBgcUx4oDe1sMjBEpza4uL7H5SE9zK1/yYQNA00qwN82AFIHAhYRJbCqaXUMcuUcdCzXedTX5H+w41J/BJ3d4t2PYnfi7tpQw5gZhjElcasjwI7d6ezubxnd7Y59sGUA2adBqwtWTNk3bH4tm/Agj7Ux9EqXXOjK0oYEGIL1dMtKeSGhOtq2BiovxWcDtjgKBYxHga0FcoHhJpebXxEHpuBCezrbdozW1VZmz1ruFr63gI63W6OtK3PLy/sIcluA3AqQ220AmZUgsqdZdcDcsg+tkmMwd8eebWPXAI3zhY7ugr1F2FsBS31ffkTh+jIFdUAS4FIkMeExdocOsUFP2gOwLQvBUaZwGJcfQWEtn8gJ2+R5HBq0If4hw2YK/9IhnmiFh45DeaZmqKeOtfxt06Rf0FnlaWN5kGfOGcwc9OQJ7Oji5lMUa7kPKOP9KVIJMvWb1fBYcjiGtsgHv0RVcxnK8qRhMxaHNLNt97xzYZVnNvDarV5v4XXnQu4aSSkl3S+eJOXgqfNqC3bBzr5yUJ2M16f0nVHSXJnTl1o7e1b3wPy6MMNwuOcdeFZpMNYiB1Ji+7XM0FM/Us/KRXZtObpZ8OAb/CZHZpwnhVCyVkmXFI4Z0stKgEUvTMqp0urDvWDZmHvrkYdqtCl6gm7WSKTwidfuftPkcE5FoguNgKnHkcszkaYD9FYzB0DkJbB5oFANmFAfasFRsQmEJZ2MLfFZFOWgr9gtTRtExBqawvJLkWUR9++vcd6KffalCP/7xwPlRhSF9F8Dr2s//fN/pj2YD21i/KvRQrsaUbi/dj1CM8DEi02sOP38x4gzzQGbiWtZPEg5CmQU0PH7BdnAqEmXbWBpGUkt21fAmp0jAVd6iUS6osNSUPU30Npzmv5rujnaDClNs5MXzjyOBMdYKEl6Ol0b1YFjI571wAKEFcYfXOSMss+sjAfigUB78wUmFqYModwx+IKicgyTw6Ko1y3vyuSu2GLL15nWJx/nbmlLd2xMv4aCVRveS9loBBq2BeNHI+7plH9QJS57zj1uT1kw4UVqnHx9+PDkBQwQ3HNIONMrkgFjNIIQhc4eOFBFRDkXTeRZKLzgTKXtlgP9dJiyONskI7lHXUJX56LwAoyy5SIHeN3AbpNx0Xu5M81+mI2M969jMFzAPXgwj3yXF4eni/fitKDafLlbpr1+GDVI3wryw6jK8IUNV4G+ZTeY2IDmNSzxJOMr+mCnyq1FbJzIOa6i6+1/FNJ9GbCO2pYOHFxVnCCfXELpNh6qltm5rpOUgYapg3b3NVqQ70QekVEK1zIN53x7qqQQ7ZkWvN8svd8q79VtLaClk4uQEF0uXUpIyQu4Vyenb5y5YHD9l9//1Z4GYIqldPauJZMAZMuf/k7elf3T/7Lxeg1dlF471o02wpvVIHOEb0cDgJ5nkR/Ar62dZPiRorh2vaQPgiBzP/+Yjgqc/vR3xZ1fOaq4LtzTu6ZGIwfLA4uL1gZN0tZ2JFh14xov26DLKWgh/coY/CIYsh2n/ANIURgRPfzq9jZKWS7bsRT63TUBtdoNZsRvoXp3VzyQ+XfZVRn/oMOnLx2A43HclQmtqizTKqMru1LzYPEjLr8WOt2cfxgB43GGXqbYm3T+I0yVK0cgRJ4N9D7HTdpskFXXvcpBhYbeIOlBMgV6Ew9166hL6ZaVMzFAZ6JoU51Yfq3R1Jb7T7iOUoQ8Sbe69k4vtbo71uGKYbLWqGruqzPb3fN2j47AbNnzHnnD4bJZcsfZztF+eZQT1s9sOv3BqlsZKhISVgolWlgslJDGoblQstjWG3P9EeqYp0U6paEXHeuKNfcSP5O4uyPaVrAIIKtEnK1YHelIb4GJuiumBTWkRJeszDM+ovu0xsiqV6w7MqPPfzWZe8UANeJ1jwrqFNXq4TKlhFT8P0kq7kfGhiYxBgfKprUDUeaUI5b50k1JqQwJ01e4vYwzGgPLN8WeVRQvSULdbFhzqSJ9VTsquViJXyppJctIMJtZOjeyk8Q/djpPEr+n9GZ9dFaHECMwcjPP0ZhPhBlvfYTnj/D80b69sHxY6YWyBndSJOLcy36gKKKztAQZnYcYvjkXhoDjFzxdEi20rkw8pC2D3OV1BOMK6Oc/QZ/B6x4c9JZ2fGnBXF2s26+m4FWwsjye42gk4XxH2Ef19VdMIr+4c9BgPwkT7jWUGnr5fU+9NO0SOpEnhS8KgWA4cLkQZWRca2STWv90aWul8wDNiFrnIjmLZDYdQFGYFM0rfVs1r5byA2iMkFaGkAj7d51lic/4hT8z7yWEKYZV+pT4FMmStYJGynl4gQxChrdQi5KDBPuoMxDHimJn1zRdsay03eX5BOafPXwo0tDwLBvmJJPszBUdmF09fAgKDD8sIjxL6ffIFLWfdWwqJn2nW4i349f7liVPSD7s16YsjOoH8yVxXWaO1+g283ODVj0u5qTuD8VMWIe0yJJqxLk2aSWSroRFmumssuYkxQNlCkYYfFoYBqnbeHwCUrW49fI8AAslf4vsQjQja58EiQH94LEJVP++lbo2fqvQbMMDHaaZWzutrtVSgP4SgYBNCEZ8lw+Atrtg8rCbIloir6rSKOIkqJxkeEdyzdqUrVDNs5HXLf3coWMv5PUMJvb9AA9pMiPMxO2w4vTxHkAWOb9SGdLuEcFVr4gDBNoUvy3Mo560cPEjkoUBS3fNAV8wY2/wJWOB77FKu1ZDHHefwAumd9JXXMsQdsDTxMCarqyxOva+2WuCXWw0LeaqLyw/E4tnYeAGT2UukdTkyLfSV1ZPOv6/Y5dx8yILnVFf6LFcAuAfVXsIL0Mnj5+EZ/VkJ3mZDm7OFXEKSrgwK8a4PjENa1YZ9av9k9gQwnlOrlRvOYwgIwX4lRJwU0klvSHAQhc1NjC3dbz2wsYxeIX1j2Dp5qIK+DYiVU21EBWJsyxDxJFNLdOsiBoXjQuLCnxNx1WI9oReztWiVveitya+UZ4DFURU2wHSnYcPcQQUphS8fBcbiXmM7tGB9LreiI2JH6zgH2KRVEu01uupazJgETr6K30byRJtb5sWnqlENfwtarw8JZDYeQ/KsK4M/VDgS8Z+xLdcelEo2lv4ic4eBnYQfdPKmR+IVzmmsBbrwaOZXtzPEAM6UdgXeBBIUjN3g+gXBKwBJyu5ML/O5xBG4pr4eRW+/2pHVDi3e9Ll/KoQ/hdQyJLVUHYQ1TwhAKl5ld3CMQoiKysoS2aB+DYc6AM+jTMMuZUrXVgJRuaPJpw2Ht5eilejF6aMzP18C7GW7ySNGcU+LFKepEWhQpOwfyLMlp60+gpgLFXA8bWsWsmcmtctSyLOc5qg5uJ0PWY3UWes0k8LgMOLrwEyEfKy3xcxbvGlu+aco7FT7o84kaGWsSKRSLjn5GroZNWC349brYyuju3owhGFfXIgPIzDcK0GULcLyyycanPT2HXjsawsfTLEemEuSB7hF/u4O0ZBXEwSTbblSRafIyg+xwWbNME/poMJT1ZaUuHuNJkN9EwpU+aHt6cSQGhDzI/BIZ4F8cA4Ty+sOX5xrSc+m0PfIGgNMBx5o+MmUpNnEFn6oI6SOYMBdRHiSlPYj6CbuGnNRQfyozOLlduOLPGfM+B05NlS0zD6+z/4RYUf3n77PA4TkP4wG/TRaPcigLgDUxy8Y0mpePX2CD/ZGV9C522651jmWF497pj5OI2vKX5FiKIurdgOsyAHPkZN5fmNDNV5PBOBy+Tzj2AaMdIgUZzzQRxf6YrZgX91yCkxxQi3H20TelmAqroDnjlYKmkCvoGjlF9tN6BdmBhVfPwPbZmA/Acb/7zQAwqfK6PXLqkiLvVuwckN2oATfTZMr+6wvvn+7J1O6gispd5cl+5j6x3wg97T8Va7L6bUprPnhQgC4pd1er8++/478GYwfcEf3hhzXMreuZyitVRL63yxMMvbmffwvuzqmuB3i3wvToGL0Llk+SQrVnvO0Zq99L2FnGwqz8ErMuRNh+dfoEYb+FV2bAofWu0RRJmTF1k6BNfDsjj9Q7b1h4imrBn2ltkmGQQ1K1OCsiKdpwjKLneI57AofRr7vBc2UIk+d5UlE/w4mK5G7YjsQlYojyFigF+4WEihQ58rUKQO7U5FvkaOkBT2Fb/JjEIompIvt5fEerGkkbS6vsPElJtiS/XA/oKJKocH9Im6WAT+hUKomWNfun+85sawDt4HIp9Objhq3uqY80Z8awwPJORH2fA2sbHbaR10tMykS0qrF74LESWPegoiyBPNggSFUVZU493Sog79Vqmn8FG9Ce2Aq8rtKL4GZhPamLI/6Gs1TQkgyK10jR1Zni5Wq72tSQkBBUB9UEq0vFQuwht6kY9bfl95+fo1fjf2wTxdujavGVgW0oEjFMoHiyAFOaRqN98LsqwM0hizoWB8gfmXbpQR/j/9+/+EisYOeZZB75RuWlLk5AWYBxW5uXeCho4j/mpC7ftn5Q3e0xh0ubIBvNyhjw+hj/uCBzmTRgbZT/ImcBEDk1nwHl26q+XRLZSo+XLC/LoUWGSoDaPakJI3v44z1iUHSXshqL6AJPc3HXVn4qMJLHXHPqzdp09zaTwvfVKg4WOXK9/os37Z1zot9VOC6ncD698JvDBhVPFJrSZEVeGFH+E0l8AFAd+A0+ODPYObHK/Swa+4SSdtFREfEbDllyFM8TkF0Zck08+9tlDdsRVfdVBuLVzYGThEBuBQ+3xDTY2QKBefW9e/jgorX4P4M9Kth6ohiC9VJg12fj682H6vPaa/JHaM6TPywocNkyY7Gf9GBVW+74e129CMLOsl3AlNxPee7PyX4aqKtmHj+KL339D+APv4jONuQAsxzCoDNKNSojWeNk+d36B8MPSpbn76JD7rgSjLb3SsW8NpTbkSMDAGSCvjyjkWba/WtYVh6HMQUuXK8cV9DnR3urrZeMGjFv+Wpk1eoA/2WoLUQwsmr7JalmX30oV4nkPBWnGuaJ4p3s810aoq/0zJ4zb9fZLHbfFnRP8vrIogwA=="}""").items():
        open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]

print("Código:", origen)
print(subprocess.run("nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
      or "SIN GPU: en Session options elige Accelerator = GPU T4 x2")
import psutil
print(f"RAM {psutil.virtual_memory().total/2**30:.0f} GB · CPU {os.cpu_count()} · "
      f"disco libre {psutil.disk_usage('/').free/2**30:.0f} GB · "
      f"/kaggle/working libre {psutil.disk_usage(W).free/2**30:.0f} GB")

## 2. Instalar (la primera vez 10-15 min)

In [ ]:
# 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.5 LTS
INSTALAR_TRIPOSG = False   # TripoSG casi no cabe en una T4 junto a Hunyuan; dejalo en False
import glob, os, shutil, subprocess, sys, time
t0 = time.time()
CACHE = "/kaggle/working/ap3d_cache"
HY3D = os.environ["AP3D_HY3D"]

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Falló: " + cmd[:120])
    return r.stdout

if "transformers" in sys.modules or "diffusers" in sys.modules:
    raise SystemExit("Ya se importó transformers/diffusers en esta sesión. Ve a Run › Restart & clear "
                     "outputs y ejecuta de nuevo las celdas 1 y 2.")

# --- sistema: librerias para Blender sin ventana (los avisos de apt no importan)
sh("apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y libxi6 libxkbcommon0 libsm6 "
   "libxrender1 libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null 2>&1",
   check=False)

# --- Python. Se deja el torch de Kaggle. Gradio 5 + transformers 4.49 + diffusers 0.32 son las
# versiones que necesita Hunyuan3D-2; huggingface_hub < 1.0 hace que encajen entre si.
# Pip mostrara "dependency conflicts" de paquetes de Kaggle que no se usan (numba, cudf, bigframes...).
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' "
   "'accelerate>=1.2,<1.6' 'huggingface_hub>=0.28,<1.0' einops omegaconf trimesh pymeshlab pygltflib "
   "xatlas rembg onnxruntime-gpu opencv-python-headless ninja pybind11 psutil hf_transfer "
   "sentencepiece protobuf scipy")
print(f"Paquetes listos ({time.time()-t0:.0f} s)")

# --- Hunyuan3D-2
if not os.path.exists(HY3D):
    sh(f"git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 {HY3D}")
sh(f"cd {HY3D} && {sys.executable} -m pip install -q -e . --no-deps")
if HY3D not in sys.path:
    sys.path.insert(0, HY3D)

import torch
assert torch.cuda.is_available(), "Sin GPU: en Session options elige Accelerator = GPU T4 x2"
cap = "%d.%d" % torch.cuda.get_device_capability(0)
os.environ["TORCH_CUDA_ARCH_LIST"] = cap
os.environ["MAX_JOBS"] = str(os.cpu_count())
if not shutil.which("nvcc") and os.path.exists("/usr/local/cuda/bin/nvcc"):
    os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ["PATH"]
    os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cc = f"{CACHE}/compilado/{tag}"
os.makedirs(cc, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import torch, {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cc}/custom_rasterizer*.whl")
    if whl:
        print("custom_rasterizer: desde la caché")
    else:
        assert shutil.which("nvcc"), "No encuentro nvcc (CUDA) para compilar custom_rasterizer"
        print("Compilando custom_rasterizer (5-8 min la primera vez)...")
        sh(f"cd {HY3D}/hy3dgen/texgen/custom_rasterizer && {sys.executable} -m pip wheel -q . "
           f"--no-build-isolation --no-deps -w {cc}")
        whl = glob.glob(f"{cc}/custom_rasterizer*.whl")
    sh(f"{sys.executable} -m pip install -q --force-reinstall --no-deps {whl[0]}")

d = f"{HY3D}/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cc}/mesh_processor*.so")
    if not so:
        suf = sh(f"{sys.executable} -c \"import sysconfig;print(sysconfig.get_config_var('EXT_SUFFIX'))\"").strip()
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC $({sys.executable} -m pybind11 --includes) "
           f"mesh_processor.cpp -o {cc}/mesh_processor{suf}")
        so = glob.glob(f"{cc}/mesh_processor*.so")
    shutil.copy(so[0], d)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f} s)")

# --- Blender 4.5 LTS (4.2 de respaldo), guardado en /kaggle/working/ap3d_cache
B = os.environ["AP3D_BLENDER"]
if not os.path.exists(B):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /tmp/b.tar.xz && tar -xf /tmp/b.tar.xz -C {CACHE} && "
                           f"rm -rf {CACHE}/blender && mv {CACHE}/blender-{ver}-linux-x64 {CACHE}/blender",
                           shell=True)
        subprocess.run("rm -f /tmp/b.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists(B):
            break

# --- TripoSG (opcional)
if INSTALAR_TRIPOSG:
    T = os.environ["AP3D_TRIPOSG"]
    if not os.path.exists(T):
        sh(f"git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG {T}", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    os.environ["FORCE_CUDA"] = "1"
    r = sh(f"{sys.executable} -m pip install -q --no-build-isolation diso", check=False)

print(sh(f"{B} -b --factory-startup --version | head -1").strip())
ok = have("custom_rasterizer") and glob.glob(f"{d}/mesh_processor*.so") and os.path.exists(B)
print(("✅" if ok else "⚠️"), f"Instalación lista en {time.time()-t0:.0f} s")

## 3. Modelos de IA

In [ ]:
# 3. Descargar los modelos y cargarlos en la GPU
CARGAR_MULTIVISTA = False   # varias fotos (espalda, perfiles): +5 GB de disco y ~6 GB de RAM
CARGAR_TEXTO_A_3D = False   # texto -> imagen -> 3D con HunyuanDiT (FLUX no cabe en una T4 con 29 GB de RAM)
import os, sys, time
sys.path.insert(0, "/root/ap3d_app"); sys.path.insert(0, os.environ["AP3D_HY3D"])
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or ""
except Exception:
    pass
if not os.environ.get("HF_TOKEN", "").startswith("hf_"):
    os.environ.pop("HF_TOKEN", None)          # un token vacio o falso bloquea las descargas publicas
os.environ["AP3D_T2I"] = "hunyuan"            # FLUX (12B) no cabe en la T4
import logging, warnings
warnings.filterwarnings("ignore")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

t0 = time.time()
P = config.detect()
print(config.describe(P))

def paso(nombre, fn):
    t = time.time()
    print(f"▶ {nombre}...", flush=True)
    try:
        r = fn()
        print(f"  ✓ {nombre} en {time.time()-t:.0f} s", flush=True)
        return r
    except Exception as e:
        print(f"  ⚠ {nombre}: {e}", flush=True)

# En la T4 solo se usan los modelos Turbo (5 pasos): los de 50 pasos no caben junto a la textura.
paso("Hunyuan3D-2 (forma Turbo + textura)", lambda: snapshot_download("tencent/Hunyuan3D-2", max_workers=16,
     allow_patterns=["hunyuan3d-dit-v2-0-turbo/*", "hunyuan3d-vae-v2-0-turbo/*",
                     "hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]))
if CARGAR_MULTIVISTA:
    paso("Hunyuan3D-2mv Turbo", lambda: snapshot_download("tencent/Hunyuan3D-2mv", max_workers=16,
         allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*", "hunyuan3d-vae-v2-mv-turbo/*"]))
if CARGAR_TEXTO_A_3D:
    paso("HunyuanDiT", lambda: snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled",
                                                 max_workers=16))

eng = get_engine()
paso("Cargar modelos en la GPU", lambda: eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=True,
                                                   want_turbo=True, want_quality=False))
if os.path.isdir(os.path.join(config.TRIPOSG_PESOS, "TripoSG")):
    paso("TripoSG", eng.load_triposg)
print(f"\nGPU {P['gpu']} · VRAM usada {torch.cuda.memory_allocated()/2**30:.1f} GB · "
      f"textura IA {'sí' if eng.paint is not None else 'NO'} · total {time.time()-t0:.0f} s")
print("Listo. Sigue con la 3b (prueba) o la 4 (página web).")

## 3b. Prueba rápida sin página web (recomendada)

In [ ]:
# 3b. Prueba corta SIN página web: confirma que todo funciona y mide tiempos
import glob, os, time
from ap3d.engine import get_engine
_img = sorted(glob.glob(os.path.join(os.environ["AP3D_HY3D"], "assets/example_images/*.png")))[0]
print("Imagen de prueba:", os.path.basename(_img))
_t = time.time(); _last = [None]
def _p(f, m):
    if m != _last[0]:
        _last[0] = m
        print(f"[{time.time()-_t:5.0f} s] {int(f*100):3d}%  {m}", flush=True)
_r = get_engine().run(_img, steps=5, octree=256, faces=40000, tex=2048, rig="auto",
                      tex_rapida=True, progress=_p)
_i = _r["info"]
print("\n======== RESULTADO (mándame esto para revisar) ========")
print(f"Listo en {time.time()-_t:.0f} s  ->  {_r['zip']}")
print(f"Tiempos: forma {_i.get('segundos_forma')} s · textura {_i.get('segundos_textura')} s · "
      f"Blender {_i.get('segundos_blender')} s")
print(f"Detectado: {(_i.get('clasificacion') or {}).get('tipo', _i.get('rig'))} · huesos {_i.get('huesos')} · "
      f"malla IA {_i.get('caras_alta')} -> {_i.get('caras')} caras")
print("Avisos:", _i.get("avisos") or "ninguno")

## 4. Página web

In [ ]:
# 4. Página web con contraseña (se abre en el celular)
CONTRASENA = ""   # vacía = se genera una al azar. Usuario: ap
import time, urllib.request, urllib.error
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)

# Espera a que la página responda a traves del tunel (evita abrirla y ver "502 Bad gateway")
estado = "sin respuesta"
for _ in range(30):
    try:
        urllib.request.urlopen(url + "/config", timeout=10)
        estado = "ok"
    except urllib.error.HTTPError as e:
        estado = "ok" if e.code in (401, 403) else f"HTTP {e.code}"   # 401 = pide contraseña: funciona
    except Exception as e:
        estado = str(e)[:60]
    if estado == "ok":
        break
    time.sleep(3)
print("Página web:", url, "✅" if estado == "ok" else f"⚠️ ({estado}; espera 1 min y recarga)")
print("Usuario:", user, "· Contraseña:", pw)
print("Deja abierta la pestaña de Kaggle: si la sesión se detiene, la página deja de funcionar.")

## 5. Descargar tus modelos

In [ ]:
# 5. Tus modelos: están en /kaggle/working/ap3d_salidas (panel Output → descargar)
import glob, os
zs = sorted(glob.glob("/kaggle/working/ap3d_salidas/*.zip"), key=os.path.getmtime, reverse=True)
for z in zs[:30]:
    print(f"{os.path.getsize(z)/2**20:7.1f} MB  {os.path.basename(z)}")
print(f"{len(zs)} modelos. Con Persistence = Files only se conservan para la próxima sesión.")

## 6. Diagnóstico

In [ ]:
# 6. (Si algo falla) memoria y estado
from ap3d.engine import get_engine
get_engine().log_mem()